# FaceForge: training a conditional DCGAN on CelebA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naman-verma-02/faceforge-gan/blob/main/notebooks/train_colab.ipynb)

This notebook trains the model end to end on a free Colab GPU and publishes it to the Hugging Face Hub,
where the live demo loads it from.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

| Step | What happens | Time on T4 |
|---|---|---|
| 1 | Clone repo, install deps | 1 min |
| 2 | Download + preprocess ~51k CelebA faces | 5–8 min |
| 3 | Train 25 epochs (FID every 5) | ~60–75 min |
| 4 | Inspect samples, loss curves, FID | 1 min |
| 5 | Push weights to Hugging Face | 1 min |

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/Naman-verma-02/faceforge-gan.git
%cd faceforge-gan
!pip install -q pyarrow huggingface_hub "torchmetrics[image]" gradio

Optional: mount Google Drive so checkpoints survive a Colab disconnect. If the runtime resets,
re-run the setup cells and add `--resume /content/drive/MyDrive/faceforge/last.pt` to the training command.

In [ ]:
USE_DRIVE = True
OUT_DIR = "runs/faceforge"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/faceforge"
print("checkpoints ->", OUT_DIR)

## 2. Data

`flwrlabs/celeba` stores CelebA as 19 parquet shards (~8.5k faces each). 6 shards (~51k faces) is a good
quality/time trade-off; raise it for better results if you have time. Faces are centre-cropped,
resized to 64×64 and cached as a single `.npy` array.

In [ ]:
from faceforge.config import Config
from faceforge.data import build_cache, CelebAFaces

NUM_SHARDS = 6
cache = build_cache("data", Config().attributes, NUM_SHARDS)
ds = CelebAFaces(cache)
print(len(ds), "faces")
print("attribute frequency:", dict(zip(Config().attributes, ds.labels.mean(0).round(3))))

In [ ]:
import matplotlib.pyplot as plt, numpy as np
idx = np.random.choice(len(ds), 16, replace=False)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, i in zip(axes.flat, idx):
    ax.imshow(ds.images[i]); ax.axis("off")
    on = [a for a, v in zip(Config().attributes, ds.labels[i]) if v]
    ax.set_title("\n".join(on) or "-", fontsize=7)
plt.suptitle("Real training faces with their attributes"); plt.show()

## 3. Train

In [ ]:
!python -m faceforge.train --epochs 25 --num-shards {NUM_SHARDS} --batch-size 128 --fid-every 5 --out-dir {OUT_DIR}

## 4. Results

In [ ]:
import json, glob
from IPython.display import Image, display
hist = json.load(open(f"{OUT_DIR}/history.json"))
ep = [h["epoch"] + 1 for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ep, [h["d_loss"] for h in hist], label="D loss")
ax[0].plot(ep, [h["g_loss"] for h in hist], label="G loss")
ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].set_title("Losses")
fid = [(h["epoch"] + 1, h["fid"]) for h in hist if "fid" in h]
if fid:
    ax[1].plot(*zip(*fid), marker="o"); ax[1].set_title("FID (lower is better)"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/curves.png", dpi=120); plt.show()

Each **column** below is one fixed noise vector *z*; each **row** switches on one attribute
(row 1: none, then Male, Smiling, Eyeglasses, Blond hair, Black hair, Young).

In [ ]:
samples = sorted(glob.glob(f"{OUT_DIR}/samples/*.png"))
for p in [samples[0], samples[len(samples) // 2], samples[-1]]:
    print(p); display(Image(p, width=520))

In [ ]:
import torch
from faceforge.inference import load_generator, generate, interpolate
G, cfg = load_generator(f"{OUT_DIR}/generator.pt", "cuda")
A = lambda *on: torch.tensor([1.0 if a in on else 0.0 for a in cfg.attributes])
combos = {"woman, smiling, blond": A("Smiling", "Blond_Hair", "Young"),
          "man, glasses": A("Male", "Eyeglasses"),
          "older man, black hair": A("Male", "Black_Hair"),
          "young woman, black hair": A("Young", "Black_Hair")}
fig, axes = plt.subplots(len(combos), 8, figsize=(14, 7.5))
for r, (name, y) in enumerate(combos.items()):
    for c, img in enumerate(generate(G, y, seed=r * 100, n=8)):
        axes[r, c].imshow(img); axes[r, c].axis("off")
    axes[r, 0].set_title(name, loc="left", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/showcase.png", dpi=120); plt.show()

m = interpolate(G, A("Male"), A("Smiling", "Blond_Hair", "Young"), 1, 2, steps=10)
plt.figure(figsize=(15, 2)); plt.imshow(np.concatenate(list(m.numpy()), 1)); plt.axis("off")
plt.title("Latent + attribute interpolation"); plt.savefig(f"{OUT_DIR}/morph.png", dpi=120); plt.show()

## 5. Publish to Hugging Face

Create a **write** token at https://huggingface.co/settings/tokens, then run the cells below.
The demo Space loads `generator.pt` from this model repo.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_REPO = "e-NV-y/faceforge-gan"   # <hf-username>/<repo-name>
!python -m faceforge.hub --repo {HF_REPO} --run {OUT_DIR}

## 6. Try the demo right here (optional)
Launches the same Gradio app that runs on Spaces, with a public share link.

In [ ]:
import os
os.environ["FACEFORGE_MODEL"] = f"{OUT_DIR}/generator.pt"
import app
app.demo.launch(share=True)

## 7. Copy result images into the GitHub README
Download `curves.png`, `showcase.png` and `morph.png` from `OUT_DIR`, put them in `assets/` in the repo
and commit. The README already references those paths.